# Project to practice - Maccabi Rapyed Tel Aviv

In [1]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr
import os

In [2]:
load_dotenv(override=True)
openai = OpenAI()

In [3]:
info_about_Maccabi = """Maccabi Rapyd Tel Aviv
A signature team in European basketball, six-time champ Maccabi enters every season aiming for greatness. Over the years, Maccabi has relied on the tried-and-true formula of complementing the best Israeli players with talented, committed stars from abroad. Tal Brody, Mickey Berkowitz, Motti Aroesti, Kevin Magee, Doron Jamchy, Earl Williams, Aulcie Perry, Sarunas Jasikevicius, Anthony Parker and Nikola Vujcic belong to the elite of all-time European basketball greats. Maccabi won its first EuroLeague title in 1977 and did so again in 1981 with Berkowitz excelling. There was a wait until 2001 to lift the SuproLeague trophy and then the next title came soon after, in 2004. Maccabi became the first team to repeat as EuroLeague champion
since Split in 1991 as Jasikevicius, Parker, Tal Burstein, Maceo Baston and Vujcic became a classic lineup in European basketball history. Maccabi got back to the championship game for the third year in a row in 2006, but CSKA stood in the way of a three-peat, and the Muscovites again triumphed in the 2008 title decider. Maccabi lost to Panathinaikos in the 2011 championship game before David Blatt led the club to glory in 2014. Since then Maccabi has not been back to the Final Four, but after making the playoffs three times in the last four years - and getting to a fifth game twice - Maccabi is again knocking on the door.
"""""

In [4]:
print(info_about_Maccabi)

Maccabi Rapyd Tel Aviv
A signature team in European basketball, six-time champ Maccabi enters every season aiming for greatness. Over the years, Maccabi has relied on the tried-and-true formula of complementing the best Israeli players with talented, committed stars from abroad. Tal Brody, Mickey Berkowitz, Motti Aroesti, Kevin Magee, Doron Jamchy, Earl Williams, Aulcie Perry, Sarunas Jasikevicius, Anthony Parker and Nikola Vujcic belong to the elite of all-time European basketball greats. Maccabi won its first EuroLeague title in 1977 and did so again in 1981 with Berkowitz excelling. There was a wait until 2001 to lift the SuproLeague trophy and then the next title came soon after, in 2004. Maccabi became the first team to repeat as EuroLeague champion
since Split in 1991 as Jasikevicius, Parker, Tal Burstein, Maceo Baston and Vujcic became a classic lineup in European basketball history. Maccabi got back to the championship game for the third year in a row in 2006, but CSKA stood in

In [5]:
name = "The head coach of Maccabi"

In [6]:
system_prompt = f"You are acting as {name}. You are answering questions on {info_about_Maccabi}'s website, \
particularly questions related to {info_about_Maccabi}'s history, Championships, top players, etc. \
Your responsibility is to represent {info_about_Maccabi} for interactions on the website as faithfully as possible. \
You are given a summary of {info_about_Maccabi}'s background and history which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future player who came across the website. \
If you don't know the answer, say so."

system_prompt += f"With this context, please chat with the user, always staying in character as {info_about_Maccabi}."


In [7]:
system_prompt

"You are acting as The head coach of Maccabi. You are answering questions on Maccabi Rapyd Tel Aviv\nA signature team in European basketball, six-time champ Maccabi enters every season aiming for greatness. Over the years, Maccabi has relied on the tried-and-true formula of complementing the best Israeli players with talented, committed stars from abroad. Tal Brody, Mickey Berkowitz, Motti Aroesti, Kevin Magee, Doron Jamchy, Earl Williams, Aulcie Perry, Sarunas Jasikevicius, Anthony Parker and Nikola Vujcic belong to the elite of all-time European basketball greats. Maccabi won its first EuroLeague title in 1977 and did so again in 1981 with Berkowitz excelling. There was a wait until 2001 to lift the SuproLeague trophy and then the next title came soon after, in 2004. Maccabi became the first team to repeat as EuroLeague champion\nsince Split in 1991 as Jasikevicius, Parker, Tal Burstein, Maceo Baston and Vujcic became a classic lineup in European basketball history. Maccabi got back 

In [8]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

In [9]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [10]:
# Create a Pydantic model for the Evaluation

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str


In [11]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {info_about_Maccabi} and is representing {info_about_Maccabi} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential player or future investor who came across the website. \
The Agent has been provided with context on {info_about_Maccabi} in the form of their summary"

evaluator_system_prompt += """
Return your evaluation ONLY as valid JSON in this format:

{
  "is_acceptable": true,
  "feedback": "your feedback here"
}
"""

In [12]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [13]:
from anthropic import Anthropic

claude = Anthropic(
    api_key=os.getenv("ANTHROPIC_API_KEY")
)

model_name = "claude-sonnet-4-5"

In [14]:
import json
import re


def evaluate(reply, message, history) -> Evaluation:

    messages = [
        {
            "role": "user",
            "content": evaluator_user_prompt(reply, message, history)
        }
    ]

    response = claude.messages.create(
        model=model_name,
        system=evaluator_system_prompt,
        messages=messages,
        max_tokens=300
    )

    answer = response.content[0].text

    print("RAW CLAUDE RESPONSE:")
    print(answer)

    # Extract JSON object from response
    match = re.search(r"\{.*\}", answer, re.DOTALL)

    if not match:
        raise ValueError("No JSON found in Claude response")

    json_text = match.group(0)

    data = json.loads(json_text)

    return Evaluation(**data)

In [15]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "Do Maccabi have champtionships?"}]
response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
reply = response.choices[0].message.content

In [16]:
reply

"Yes, Maccabi Rapyd Tel Aviv is a prestigious team in European basketball and has a rich history of success. They are six-time EuroLeague champions, having won their first title in 1977 and then again in 1981. The team also won the SuproLeague trophy in 2001 and added another EuroLeague title in 2004. Maccabi's legacy includes becoming the first team to repeat as EuroLeague champions since Split in 1991. Their championship pedigree is a testament to the hard work and talent that has been part of the organization's culture throughout the years."

In [17]:
evaluate(reply, "Do Maccabi have champtionships?", messages[:1])

RAW CLAUDE RESPONSE:
```json
{
  "is_acceptable": false,
  "feedback": "The response is incomplete and fails to mention all six championships that Maccabi has won. According to the provided context, Maccabi won EuroLeague titles in 1977, 1981, 2001, 2004, 2005, and 2014. The Agent only mentions four titles (1977, 1981, 2001, and 2004) and omits the 2005 and 2014 championships. The context specifically states they became 'the first team to repeat as EuroLeague champion since Split in 1991' which refers to back-to-back wins in 2004 and 2005, and mentions that 'David Blatt led the club to glory in 2014.' For a head coach representing the team and speaking to potential players or investors, providing incomplete information about the team's championship history is unacceptable. The response should accurately list all six titles to properly represent the team's prestigious legacy."
}
```


Evaluation(is_acceptable=False, feedback="The response is incomplete and fails to mention all six championships that Maccabi has won. According to the provided context, Maccabi won EuroLeague titles in 1977, 1981, 2001, 2004, 2005, and 2014. The Agent only mentions four titles (1977, 1981, 2001, and 2004) and omits the 2005 and 2014 championships. The context specifically states they became 'the first team to repeat as EuroLeague champion since Split in 1991' which refers to back-to-back wins in 2004 and 2005, and mentions that 'David Blatt led the club to glory in 2014.' For a head coach representing the team and speaking to potential players or investors, providing incomplete information about the team's championship history is unacceptable. The response should accurately list all six titles to properly represent the team's prestigious legacy.")

In [18]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

In [19]:
def chat(message, history):

    if "lebron" in message.lower():

        system = system_prompt + """

Lebron James is not part of the team and I can't give more information
"""

    else:
        system = system_prompt

    messages = (
        [{"role": "system", "content": system}]
        + history
        + [{"role": "user", "content": message}]
    )

    response = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages
    )

    reply = response.choices[0].message.content

    evaluation = evaluate(reply, message, history)

    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)

        reply = rerun(
            reply,
            message,
            history,
            evaluation.feedback
        )

    return reply

In [20]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
